# tw-med-llm-qlora — optional GGUF Q4_K_M export

This Linux Colab notebook is **not required** for adapter delivery. It is
disabled in both repository config and the first code cell. After explicit
approval, use an A100-class runtime and execute from the top. It never pushes
a model; output stays in Drive for later Windows Ollama validation.


In [ ]:
# OPTIONAL EXPORT HARD GATE — this is the only cell that may be edited.
CONFIG_EXPORT_ENABLED = False
ENABLE_GGUF_EXPORT = False
GGUF_EXPORT_APPROVAL = ""
REQUIRED_GGUF_EXPORT_APPROVAL = "GGUF_Q4_K_M_APPROVED_20260722"

if not CONFIG_EXPORT_ENABLED:
    raise RuntimeError(
        "Repo config keeps optional GGUF export disabled. Obtain explicit approval, "
        "update configs/project.toml, and rebuild this notebook first."
    )
if not ENABLE_GGUF_EXPORT or GGUF_EXPORT_APPROVAL != REQUIRED_GGUF_EXPORT_APPROVAL:
    raise RuntimeError(
        "GGUF export is locked. Set ENABLE_GGUF_EXPORT=True and copy the exact "
        "approval code in this cell only after the repo gate is enabled."
    )
print("GGUF Q4_K_M export gate passed.")


## 1. Install the pinned Phase 3-compatible export stack


In [ ]:
%pip install --quiet \
    "unsloth[colab-new]==2026.7.4" \
    "unsloth-zoo==2026.7.4" \
    "transformers==4.56.2" \
    "peft==0.19.1" \
    "bitsandbytes==0.49.2" \
    "accelerate==1.14.0" \
    "huggingface-hub==0.35.3" \
    "sentencepiece==0.2.2" \
    "protobuf==6.33.5"


## 2. Verify A100, Drive, token, and disk


In [ ]:
# ruff: noqa: E501
import hashlib
import importlib.metadata
import json
import shutil
from datetime import UTC, datetime
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import torch
from google.colab import drive, userdata

PROJECT_CONFIG = json.loads('{"data": {"medqa": {"config": "med_qa_tw_source", "dataset_id": "bigbio/med_qa", "expected_test_rows": 1413, "expected_train_rows": 11248, "expected_validation_rows": 1409, "revision": "e04abdc0672c54547fa1dbe36cfefc000e4f2657", "sample_size": 5, "source_sha256": {"test": "799f6c7881d50b1101f168f31129aea699fec67401e2b8dc9464f1f84dc40c04", "train": "b2cb4b82a7c7ffc6f6f99465197f4dfc94b04464cf2ab517a67418641c617646", "validation": "06c2334173f17ed0665088e920bbf0fb54bb64ef7432885747668235ed54701a"}}, "tmmluplus": {"dataset_id": "ikala/tmmluplus", "revision": "81d53e38340c9ade988f7fed8996da6554b504f3"}}, "evaluation": {"bootstrap_iterations": 10000, "calibration_examples": 20, "control_subjects": ["chinese_language_and_literature", "geography_of_taiwan", "logic_reasoning", "computer_science", "general_principles_of_law"], "forgetting_margin_percentage_points": 2.0, "full_approval": {"approval_phrase": "確認解鎖 Phase 4 正式評估", "approved": true, "approved_at": "2026-07-22", "approved_requests": 28758, "calibration_manifest_sha256": "d9c0f23e72808f0a8fc4edc1a7889719637f9390346725d5f72f10c4d3e2cdf2", "calibration_run_id": "20260722T061028Z", "calibration_validation_sha256": "6c39aff8cd4e82b80d24a4ce7f7959e7b7fd5ba9546f2432ef3b7c96212b2d85", "compute_units_per_hour": 5.3, "parallel_workers": 4, "projected_compute_units": 15.791642498050743, "projected_compute_units_with_20pct_buffer": 18.94997099766089, "projected_hours": 2.979555188311461, "required_approval_code": "PHASE4_FULL_28758_APPROVED_20260722", "shard_size": 250}, "full_shuffle_seed": 3407, "generation": {"max_tokens": 256, "minimum_calibration_parse_rate": 0.8, "temperature": 0.0, "token_limit_hits_count_as_incorrect": true, "token_limit_hits_fail_calibration": false, "top_p": 1.0}, "medical_subjects": ["basic_medical_science", "clinical_psychology", "dentistry", "occupational_therapy_for_psychological_disorders", "optometry", "pharmacology", "pharmacy", "traditional_chinese_medicine_clinical_medicine"], "phase3_adapter": {"archive_bytes": 113079186, "archive_sha256": "2c537dfd3049319286c678a3ca3aa72e3f20baa7e0f44bde93ff7ee4dc47e43e", "base_model_id": "taide/Gemma-3-TAIDE-12b-Chat-2602", "drive_archive": "/content/drive/MyDrive/tw-med-llm-qlora/phase3/141226999eacb67ffd9a/full/runs/20260722T014216Z-phase3-full.zip", "selected_checkpoint": 700}, "stability_examples_per_subject": 100, "stability_seeds": [3407, 3408, 3409], "twinkle_eval": {"extractor": "box", "repository": "https://github.com/ai-twinkle/Eval", "revision": "470bbec130fa95c9e2e06c6a4b06db4a51390a06", "scorer": "exact_match", "version": "2.8.0"}, "vllm": {"expected_torch_cuda": "12.9", "gpu_memory_utilization": 0.85, "language_model_only": true, "max_lora_rank": 16, "max_model_length": 2048, "quantization": "bitsandbytes", "torch_backend": "cu129", "uv_version": "0.11.31", "version": "0.25.1", "wheel_sha256": "9e206f370c934a2d4b6b1f05d3d09708d344e05d80260189ef19f60755709431", "wheel_url": "https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl", "wheel_variant": "cu129"}, "workload": {"expected_medqa_full_requests": 4239, "expected_tmmlu_full_requests": 16719, "expected_tmmlu_stability_requests": 7800, "expected_total_requests": 28758, "full_model_count": 3, "medqa_test_rows": 1413, "stability_model_count": 2, "tmmlu_test_rows": 5573}}, "export": {"gguf": {"enabled": false, "quantization_method": "q4_k_m", "run_environment": "colab_linux"}}, "hardware_profiles": [{"allow_tf32": true, "batch_size": 8, "gradient_accumulation_steps": 2, "max_sequence_length": 2048, "min_compute_capability": [8, 0], "min_vram_gib": 70.0, "model_profile": "primary", "name": "primary_80g", "requires_bf16": true}, {"allow_tf32": true, "batch_size": 4, "gradient_accumulation_steps": 4, "max_sequence_length": 2048, "min_compute_capability": [8, 0], "min_vram_gib": 38.0, "model_profile": "primary", "name": "primary_40g", "requires_bf16": true}, {"allow_tf32": true, "batch_size": 1, "gradient_accumulation_steps": 16, "max_sequence_length": 2048, "min_compute_capability": [8, 0], "min_vram_gib": 22.0, "model_profile": "primary", "name": "primary_24g", "requires_bf16": true}, {"allow_tf32": false, "batch_size": 2, "gradient_accumulation_steps": 8, "max_sequence_length": 2048, "min_compute_capability": [7, 5], "min_vram_gib": 14.0, "model_profile": "fallback", "name": "fallback_16g", "requires_bf16": false}], "inference": {"phase3_adapter": {"archive_sha256": "2c537dfd3049319286c678a3ca3aa72e3f20baa7e0f44bde93ff7ee4dc47e43e", "base_model_id": "taide/Gemma-3-TAIDE-12b-Chat-2602", "base_model_revision": "4de0b93b99f8b61b59c40d019fd593bdd1c42249", "selected_checkpoint": 700}, "windows_4090": {"attention_implementation": "sdpa", "double_quantization": true, "load_in_4bit": true, "max_new_tokens": 64, "minimum_compute_capability": [8, 9], "minimum_vram_gib": 22.0, "ollama_model_name": "tw-med-taide-12b", "quantization_type": "nf4", "required_gpu_name": "RTX 4090", "requires_bf16": true}}, "models": {"fallback": {"baseline_id": "google/gemma-3-4b-it", "baseline_revision": "093f9f388b31de276ce2de164bdc2081324b9767", "batch_size": 2, "gradient_accumulation_steps": 8, "max_sequence_length": 2048, "model_id": "twinkle-ai/gemma-3-4B-T1-it", "requires_bf16": false, "revision": "63b2bb819a7885b476c2ff98a418de8400661f9d"}, "primary": {"baseline_id": "google/gemma-3-12b-it", "baseline_revision": "96b6f1eccf38110c56df3a15bffe176da04bfd80", "batch_size": 1, "gradient_accumulation_steps": 16, "max_sequence_length": 2048, "model_id": "taide/Gemma-3-TAIDE-12b-Chat-2602", "requires_bf16": true, "revision": "4de0b93b99f8b61b59c40d019fd593bdd1c42249"}}, "project": {"effective_batch_size": 16, "seed": 3407}, "publication": {"adapter_repo_id": "", "enabled": false, "github_repository_url": "https://github.com/kuotunyu/tw-med-llm-qlora", "requires_explicit_repo_id": true, "requires_explicit_visibility_confirmation": true, "visibility": "private"}, "training": {"eval_steps": 100, "learning_rate": 5e-05, "load_in_4bit": true, "lora_alpha": 16, "lora_dropout": 0.0, "lora_rank": 16, "lr_scheduler_type": "cosine", "num_train_epochs": 1, "optimizer": "adamw_8bit", "save_steps": 100, "save_total_limit": 2, "smoke_examples": 100, "smoke_steps": 10, "warmup_ratio": 0.03}}')
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Colab Secret HF_TOKEN is required")
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required")
properties = torch.cuda.get_device_properties(0)
total_vram_gib = properties.total_memory / 1024**3
gpu = {
    "name": properties.name,
    "total_vram_gib": total_vram_gib,
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "bf16_supported": bool(torch.cuda.is_bf16_supported()),
}
if total_vram_gib < 38 or not gpu["bf16_supported"]:
    raise RuntimeError(f"A100-class BF16 GPU with at least 38 GiB is required: {gpu}")
free_disk_gib = shutil.disk_usage("/content").free / 1024**3
if free_disk_gib < 100:
    raise RuntimeError(f"At least 100 GiB free local disk is required: {free_disk_gib:.1f}")
drive.mount("/content/drive")
phase3 = PROJECT_CONFIG["evaluation"]["phase3_adapter"]
export_config = PROJECT_CONFIG["export"]["gguf"]
archive_path = Path(phase3["drive_archive"])
local_root = Path("/content/tw-med-gguf-export")
adapter_dir = local_root / "adapter"
export_dir = local_root / "q4_k_m"
drive_output = Path("/content/drive/MyDrive/tw-med-llm-qlora/phase5/gguf-q4-k-m")
for directory in (local_root, adapter_dir, export_dir):
    directory.mkdir(parents=True, exist_ok=True)
print(json.dumps({"gpu": gpu, "free_disk_gib": free_disk_gib}, indent=2))


## 3. Verify and extract only the selected step-700 adapter


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not archive_path.is_file():
    raise FileNotFoundError(f"Phase 3 full archive not found: {archive_path}")
archive_sha256 = sha256_file(archive_path)
if archive_sha256 != phase3["archive_sha256"]:
    raise RuntimeError(
        f"Phase 3 archive SHA-256 mismatch: {archive_sha256} != "
        f"{phase3['archive_sha256']}"
    )
with ZipFile(archive_path) as archive:
    selected = []
    for member in archive.infolist():
        normalized = PurePosixPath(member.filename.replace("\\", "/"))
        if normalized.is_absolute() or ".." in normalized.parts:
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        if normalized.parts and normalized.parts[0] == "adapter" and not member.is_dir():
            selected.append((member, PurePosixPath(*normalized.parts[1:])))
    names = {relative.as_posix() for _, relative in selected}
    required = {"adapter_config.json", "adapter_model.safetensors"}
    if not required.issubset(names):
        raise RuntimeError(f"Phase 3 archive is missing adapter files: {required - names}")
    for member, relative in selected:
        target = adapter_dir.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open("wb") as output:
            shutil.copyfileobj(source, output)
adapter_config = json.loads(
    (adapter_dir / "adapter_config.json").read_text(encoding="utf-8")
)
model_config = PROJECT_CONFIG["models"]["primary"]
if adapter_config.get("base_model_name_or_path") != model_config["model_id"]:
    raise RuntimeError("Adapter/base mismatch; export stopped")
print({"adapter_files": len(selected), "archive_sha256_verified": True})


## 4. Load the pinned base and frozen adapter


In [ ]:
from peft import PeftModel
from unsloth import FastModel

base_model, processor = FastModel.from_pretrained(
    model_name=model_config["model_id"],
    revision=model_config["revision"],
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    token=HF_TOKEN,
)
model = PeftModel.from_pretrained(
    base_model,
    str(adapter_dir),
    is_trainable=False,
    low_cpu_mem_usage=True,
)
model.eval()
trainable, total = model.get_nb_trainable_parameters()
if trainable != 0 or total <= 0:
    raise RuntimeError(f"Frozen adapter audit failed: trainable={trainable}, total={total}")
if not hasattr(model, "save_pretrained_gguf"):
    raise RuntimeError(
        "Current Unsloth no longer exposes save_pretrained_gguf on this PEFT model; "
        "stop and recheck the official API instead of using an unverified fallback."
    )
print({"adapter_reloaded": True, "trainable_parameters": trainable})


## 5. Export Q4_K_M and an Ollama Modelfile


In [ ]:
model.save_pretrained_gguf(
    str(export_dir),
    tokenizer=processor,
    quantization_method="q4_k_m",
    maximum_memory_usage=0.5,
)
gguf_files = sorted(export_dir.glob("*.gguf"))
if len(gguf_files) != 1 or gguf_files[0].stat().st_size <= 0:
    raise RuntimeError(f"Expected exactly one non-empty GGUF file: {gguf_files}")
gguf_path = gguf_files[0]
modelfile = export_dir / "Modelfile"
modelfile.write_text(
    "\n".join(
        [
            f"FROM ./{gguf_path.name}",
            "PARAMETER temperature 0",
            "PARAMETER seed 3407",
            "PARAMETER num_ctx 2048",
            "",
        ]
    ),
    encoding="utf-8",
)
drive_output.mkdir(parents=True, exist_ok=True)
copied = {}
for source in (gguf_path, modelfile):
    partial = drive_output / (source.name + ".partial")
    destination = drive_output / source.name
    shutil.copy2(source, partial)
    if sha256_file(partial) != sha256_file(source):
        raise RuntimeError(f"Drive copy hash mismatch: {source.name}")
    partial.replace(destination)
    copied[source.name] = {
        "path": str(destination),
        "sha256": sha256_file(destination),
        "bytes": destination.stat().st_size,
    }
receipt = {
    "schema_version": 1,
    "phase": 5,
    "optional_export": "gguf_q4_k_m",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "base_model_id": model_config["model_id"],
    "base_model_revision": model_config["revision"],
    "adapter_checkpoint": int(phase3["selected_checkpoint"]),
    "phase3_archive_sha256": archive_sha256,
    "gpu": gpu,
    "packages": {
        package: importlib.metadata.version(package)
        for package in ("unsloth", "unsloth-zoo", "transformers", "peft")
    },
    "files": copied,
    "chat_template_warning": "Validate the exported GGUF in Ollama before use.",
    "published": False,
}
receipt_path = drive_output / "gguf-export-receipt.json"
receipt_path.write_text(json.dumps(receipt, ensure_ascii=False, indent=2) + "\n")
print(json.dumps(receipt, ensure_ascii=False, indent=2))
print("Optional export complete. No model was uploaded or published.")
